
This is a code for transforming de data from full_load.raw into a unique table for the next layer of data: the bronze layer for the CLIENTS data.



In [0]:
df_full = spark.read.format("parquet").load("/Volumes/raw/sistema_pontos/full_load/clientes/",sep=";")
df_full.coalesce(1).write.format("delta").saveAsTable("bronze.point_system.clients")

In [0]:
df_cdc = spark.read.format("parquet").load("/Volumes/raw/sistema_pontos/cdc/clientes/",sep=";")
df_cdc.createOrReplaceTempView("customers")

query = "SELECT * FROM customers QUALIFY ROW_NUMBER() OVER (PARTITION BY idCliente ORDER BY DtAtualizacao DESC) = 1"

df_cdc_unique = spark.sql(query)

In [0]:
import delta

bronze = delta.DeltaTable.forName(spark,'bronze.point_system.clients')

(bronze.alias("b") 
    .merge(df_cdc_unique.alias("d"),
    "b.idCliente = d.idCliente") 
    .whenMatchedDelete(condition = "d.flOperation = 'D'")     
    .whenMatchedUpdateAll(condition = "d.flOperation ='U'") 
    .whenNotMatchedInsertAll(condition = "d.flOperation = 'I' OR d.flOperation = 'U'")
    .execute() 
    )
